# #Stoichemoetry

In [1]:
%%capture 
!pip install chemparse
!pip install mendeleev

In [2]:
import mendeleev
from mendeleev import element
from mendeleev import get_all_elements
import numpy as np
import sympy as sp
import pandas as pd
import chemparse

SnO2 + H2 -> Sn + H2O

KOH + H3PO4 -> K3PO4 + H2O

In [ ]:
def solve_stoichemeotry(equation):
  reactants_str, products_str = equation.split('->', 1)
  all_compounds = [c.strip() for c in equation.replace("->", "+").split("+")]
  
  elements_present = set()
  for compound in all_compounds:
      if compound:
          parsed_data = chemparse.parse_formula(compound)
          elements_present.update(parsed_data.keys())
  
  sorted_elements = sorted(list(elements_present))
  element_to_index = {element: i for i, element in enumerate(sorted_elements)}
  
  compound_coefficient_vectors = []

  for compound_full in all_compounds:
      compound = compound_full.strip()
      if compound:
          parsed_data = chemparse.parse_formula(compound)
          
          # Create a vector for the current compound based on sorted_elements
          coefficient_vector = np.zeros(len(sorted_elements))
          for element, count in parsed_data.items():
              coefficient_vector[element_to_index[element]] = count

          # Check if the compound is a product and apply scaling
          if compound_full.strip() in [c.strip() for c in products_str.split('+')]:
              scaled_vector = coefficient_vector * -1
              compound_coefficient_vectors.append(scaled_vector) 
          else:
              compound_coefficient_vectors.append(coefficient_vector)
          
  large_matrix = np.vstack(compound_coefficient_vectors).T # Transpose to get elements as rows, compounds as columns
  print()
  print(f"Elements in rows (sorted alphabetically): {sorted_elements}")
  print(f"Compounds in columns: {all_compounds}")
  print("Initial large matrix (elements as rows, compounds as columns):")
  print(large_matrix)
  
  rref_matrix = sp.Matrix(large_matrix).rref()[0]
  np_rref = np.array(rref_matrix).astype(np.float64)
  print()
  print("RREF of the large matrix:")
  print(np_rref)

  # Extract the last column (which represents the dependence on the free variable)
  last_col_rref_np = np_rref[:, -1]

  # Construct the vector of coefficients (including the free variable coefficient of 1)
  # The signs are flipped because of the RREF interpretation for Ax = 0 solutions
  stoich_raw_np = np.append(-last_col_rref_np, 1.0)
  
  # Convert to sympy Rational for precise fraction handling
  stoich_raw_sympy = [sp.Rational(x) for x in stoich_raw_np]

  # Get denominators
  denominators = [x.q for x in stoich_raw_sympy] 

  # Calculate LCM of denominators
  scaling_factor = sp.lcm(denominators)

  # Scale the raw coefficients to get integer coefficients
  final_coefficients_sympy = [x * scaling_factor for x in stoich_raw_sympy]
  
  # Convert back to numpy array of integers
  final_coefficients_np = np.array([int(x) for x in final_coefficients_sympy])

  # Reshape to a column vector (Nx1 matrix)
  stoichiometric_matrix = final_coefficients_np.reshape(-1, 1)
  
  print()
  print("Stoichiometric coefficients (scaled to whole numbers):")
  # Create a DataFrame to display compounds next to their coefficients
  balanced_equation_df = pd.DataFrame({
      'Compound': all_compounds,
      'Coefficient': stoichiometric_matrix.flatten()
  })
  display(balanced_equation_df)

solve_stoichemeotry('Fe + O2 -> Fe2O3')

In [3]:
def grams_to_mol(grams, element):
    element = mendeleev.element(element)
    mols = grams / element.atomic_weight
    return mols
grams_to_mol(100, 'C')

8.32570144034635